# Text Preprocessing Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Peel the packaging.** Four regex/library swabs take scraped HTML down to plain sentences.

In [ ]:
import html
import re

scraped = ("<p>Screen cracked within <b>two days</b>!</p> "
           "Details: http://shop.example.com/r42 &amp; photos.")

step1 = re.sub(r"<[^>]+>", " ", scraped)              # 1. strip tags
step2 = re.sub(r"https?://\S+|www\.\S+", " ", step1)  # 2. strip URLs
step3 = html.unescape(step2)                          # 3. decode &amp; ->
clean = re.sub(r"\s+", " ", step3).strip()            # 4. collapse whitespace

print(repr(clean))
# prints: Screen cracked within two days ! Details: & photos.

**2. Three tokenizers.** Each strategy fixes the previous blind spot: glued punctuation, shredded apostrophes, then both handled.

In [ ]:
import re

sentence = "Don't panic: the refund arrived, and it's correct!"

print(sentence.split())
# fastest, crudest - "correct!" stays glued to its bang
print(re.findall(r"\w+", sentence))
# Don/t torn apart: "don" + "t"
print(re.findall(r"\w+(?:'\w+)?|[^\w\s]", sentence))
# keeps don't / it's whole AND separates standalone punctuation - our pick

**3. Expand contractions.** Dict values are lists because one contraction becomes several words; unmatched tokens pass through unchanged.

In [ ]:
CONTRACTIONS = {
    "don't": ["do", "not"], "won't": ["will", "not"], "isn't": ["is", "not"],
    "wouldn't": ["would", "not"], "it's": ["it", "is"], "i'm": ["i", "am"],
}

def expand(tokens):
    out = []
    for tok in tokens:
        out.extend(CONTRACTIONS.get(tok, [tok]))
    return out

demo = "don't worry: it's working, she wouldn't lie"
print(expand(demo.split()))
# ['do', 'not', 'worry:', 'it', 'is', 'working,', 'she', 'would', 'not', 'lie']
# 'worry:' survived because .split() left the colon glued on -
# that is why real pipelines tokenize BEFORE expanding.

## Part 2 — Practice

**4. Case decisions.** Lowercasing merges variants but erases entity/emotion signal; `casefold()` is the aggressive matcher's tool.

In [ ]:
pair_a = "I work at Apple."
pair_b = "I eat an apple daily."

print(pair_a.lower())   # i work at apple. - company sense lost
print(pair_b.lower())   # fine here

shout = "NOT HAPPY WITH THIS"
print(shout, "->", shout.isupper())   # True - casing carries emotion

print("Straße".lower() == "STRASSE".lower())        # False - lower() cannot help
print("Straße".casefold(), "|", "STRASSE".casefold())
print("Straße".casefold() == "STRASSE".casefold())  # True

**5. The stopword trap.** Removing `not` erases negation — the canonical way helpful defaults destroy sentiment tasks.

In [ ]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so",
    "is", "am", "are", "was", "were", "be", "been",
    "this", "that", "it", "its", "i", "you", "we", "they",
    "my", "your", "our", "me", "him", "them",
    "not", "no", "never", "very", "just", "again", "only",
}

positive = "this phone is good"
negative = "this phone is not good"

strip_all = lambda s: [w for w in s.split() if w not in STOPWORDS]

print(strip_all(positive))   # ['phone', 'good']
print(strip_all(negative))   # ['phone', 'good'] - IDENTICAL! sentiment erased

NEGATION_KEEP = {"not", "no", "never"}
safe_stopwords = STOPWORDS - NEGATION_KEEP

strip_safe = lambda s: [w for w in s.split() if w not in safe_stopwords]
print(strip_safe(negative))  # ['phone', 'not', 'good'] - meaning survives

**6. Build a mini stemmer.** Rule order matters; the length guard stops silly stems. Overstemming merges unrelated worlds.

In [ ]:
STEM_RULES = [
    ("ational", "ate"), ("tional", "tion"),
    ("ities", ""), ("ity", ""),
    ("ies", "y"),
    ("ingly", ""), ("ing", ""),
    ("ied", "y"), ("ed", ""),
    ("ly", ""), ("ness", ""), ("ment", ""),
    ("es", ""), ("s", ""),           # must come AFTER "ies"
]

def mini_stem(word):
    for suffix, replacement in STEM_RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: len(word) - len(suffix)] + replacement
    return word

for w in ["playing", "studies", "scratches", "studied", "university", "universes"]:
    print(f"{w:>12} -> {mini_stem(w)}")
# play / study / scratch / study / univers / univers
# Overstemming: university and universes collapse into one non-word,
# merging academia with outer space. Cheap, but violent.

**7. Lemmatize by lookup.** The dictionary rescues what no suffix rule can reach; regular forms still get stemmed.

In [ ]:
LEMMA_MAP = {
    "ran": "run", "went": "go", "arrived": "arrive",
    "mice": "mouse", "better": "good", "happily": "happy",
}

def lemmatize(word):
    return LEMMA_MAP.get(word, mini_stem(word))

words = ["ran", "better", "mice", "studied", "arrived", "happily"]
for w in words:
    print(f"{w:>10} -> stem: {mini_stem(w):<8} lemma: {lemmatize(w)}")
# ran needs the LOOKUP (no suffix to chop); studied falls through to rules

## Part 3 — Challenge

**8. Compose the pipeline.** One entry point, risky choices as explicit flags — the pipeline is now testable and identical everywhere.

In [ ]:
import re

CONTRACTIONS = {
    "don't": ["do", "not"], "doesn't": ["does", "not"], "didn't": ["did", "not"],
    "can't": ["can", "not"], "won't": ["will", "not"], "isn't": ["is", "not"],
    "aren't": ["are", "not"], "wasn't": ["was", "not"], "haven't": ["have", "not"],
    "hasn't": ["has", "not"], "couldn't": ["could", "not"], "wouldn't": ["would", "not"],
    "shouldn't": ["should", "not"], "it's": ["it", "is"], "i'm": ["i", "am"],
    "i've": ["i", "have"], "they're": ["they", "are"], "we're": ["we", "are"],
    "you're": ["you", "are"], "let's": ["let", "us"], "that's": ["that", "is"],
}
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "because",
    "of", "at", "by", "for", "with", "about", "to", "from", "in", "on",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had", "will", "would", "can", "could",
    "this", "that", "these", "those", "it", "its",
    "i", "you", "he", "she", "we", "they",
    "my", "your", "his", "her", "their", "our", "me", "him", "them",
    "not", "no", "never", "very", "just", "again", "only",
}
NEGATION_KEEP = {"not", "no", "never"}

STEM_RULES = [("ational", "ate"), ("tional", "tion"), ("ities", ""), ("ity", ""),
              ("ies", "y"), ("ingly", ""), ("ing", ""), ("ied", "y"), ("ed", ""),
              ("ly", ""), ("ness", ""), ("ment", ""), ("es", ""), ("s", "")]

def mini_stem(word):
    for suffix, replacement in STEM_RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: len(word) - len(suffix)] + replacement
    return word

LEMMA_MAP = {
    "ran": "run", "felt": "feel", "went": "go", "bought": "buy",
    "arrived": "arrive", "mice": "mouse", "better": "good", "best": "good",
    "happily": "happy",
}

def lemmatize(word):
    return LEMMA_MAP.get(word, mini_stem(word))

def preprocess(text, remove_stops=True, keep_negations=True):
    text = re.sub(r"<[^>]+>", " ", text)                    # 1. clean: tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)      #    clean: URLs
    text = re.sub(r"[^a-z0-9' ]+", " ", text.lower())       #    clean: noise
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)        # 2. tokenize
    tokens = [t for tok in tokens for t in CONTRACTIONS.get(tok, [tok])]  # 3. expand
    if remove_stops:                                        # 4. filter
        stops = STOPWORDS - NEGATION_KEEP if keep_negations else STOPWORDS
        tokens = [t for t in tokens if t not in stops]
    return [lemmatize(t) for t in tokens]                   # 5. lemmatize

print(preprocess("<p>I DIDN'T like it!</p> Visit http://x.example.com"))
# ['not', 'like', 'visit'] - cleaned, expanded, filtered, lemmatised

review = "The battery ISN'T terrible, but it WON'T survive a full day."
print(preprocess(review, keep_negations=True))
print(preprocess(review, keep_negations=False))
# Flip the flag and BOTH negations vanish silently: 'terrible' suddenly
# reads positive. For sentiment tasks, protect negations.

**9. Before and after.** Raw vs processed side by side makes the trade visible: fewer, cleaner, more consistent tokens.

In [ ]:
DOCS_MIX = [
    "<p>I DIDN'T expect much, but this keyboard is GREAT!</p>",
    "Track your parcel anytime at http://help.example.com/track.",
    "It's been nine days and my order hasn't arrived.",
    "Setup was easy and the app doesn't crash anymore.",
]

processed = [" ".join(preprocess(doc)) for doc in DOCS_MIX]

for raw, proc in zip(DOCS_MIX, processed):
    print(f"{raw[:44]:<46} -> {proc}")
# <p>I DIDN'T expect much, but this keyboard is -> not expect much keyboard great
# Track your parcel anytime at http://help.exam -> track parcel anytime
# It's been nine days and my order hasn't arriv -> nine day order not arrive
# Setup was easy and the app doesn't crash anym -> setup easy app not crash anymore

raw_vocab = {w.strip(".,!?<>") for d in DOCS_MIX for w in d.lower().split()}
proc_vocab = set(" ".join(processed).split())
print("unique raw tokens:", len(raw_vocab), "-> processed:", len(proc_vocab))
# Case folding, stopword removal and lemmas shrink the vocabulary hard -
# same idea, far fewer columns for the model to learn.